# Custom PyTorch model → real inference → `.onnx`

1. Define a small CV model (a U-Net not present in the service catalogue).
2. Run real inference on a folder of images and measure wall time (pure PyTorch — no repo imports).
3. Export the model to **ONNX**.

> Why ONNX and not `torch.save(model, ...)`?
> A pickled `.pt` saves a *reference* to `MiniUNet` in `__main__`. When you upload it to Streamlit, its process has no way to find that class and `torch.load` fails. ONNX bakes the graph into the file itself — no class lookup needed.

After running, upload the resulting `mini_unet_v1.onnx` through the **Models** tab in the Streamlit app (`make ui`) — it'll appear in the Predict / Recommend dropdowns and you can compare the service's prediction against the actual time measured here.

In [16]:
import statistics
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from PIL import Image

## 1. Custom model

A 4-stage U-Net (encoder–decoder with skip connections). No equivalent is registered in `src/runners/` so the service has never seen this architecture before.

In [17]:
class DoubleConv(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout),
            nn.SiLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout),
            nn.SiLU(inplace=True),
        )
    def forward(self, x): return self.net(x)


class MiniUNet(nn.Module):
    def __init__(self, in_ch=3, num_classes=21, base=32):
        super().__init__()
        c = [base, base*2, base*4, base*8, base*16]
        self.d1, self.d2, self.d3, self.d4 = (
            DoubleConv(in_ch, c[0]),
            DoubleConv(c[0], c[1]),
            DoubleConv(c[1], c[2]),
            DoubleConv(c[2], c[3]),
        )
        self.bot = DoubleConv(c[3], c[4])
        self.pool = nn.MaxPool2d(2)
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.r4 = nn.Conv2d(c[4], c[3], 1)
        self.u4 = DoubleConv(c[4], c[3])
        self.r3 = nn.Conv2d(c[3], c[2], 1)
        self.u3 = DoubleConv(c[3], c[2])
        self.r2 = nn.Conv2d(c[2], c[1], 1)
        self.u2 = DoubleConv(c[2], c[1])
        self.r1 = nn.Conv2d(c[1], c[0], 1)
        self.u1 = DoubleConv(c[1], c[0])
        self.head = nn.Conv2d(c[0], num_classes, 1)

    def forward(self, x):
        x1 = self.d1(x)
        x2 = self.d2(self.pool(x1))
        x3 = self.d3(self.pool(x2))
        x4 = self.d4(self.pool(x3))
        b  = self.bot(self.pool(x4))
        u4 = self.u4(torch.cat([self.r4(self.up(b)),  x4], dim=1))
        u3 = self.u3(torch.cat([self.r3(self.up(u4)), x3], dim=1))
        u2 = self.u2(torch.cat([self.r2(self.up(u3)), x2], dim=1))
        u1 = self.u1(torch.cat([self.r1(self.up(u2)), x1], dim=1))
        return self.head(u1)


model = MiniUNet().eval()
n_params = sum(p.numel() for p in model.parameters())
print(f'MiniUNet built — {n_params/1e6:.2f} M params')

MiniUNet built — 7.24 M params


## 2. Real inference + timing

Runs the model on real images and prints per-batch wall time. Vary the three knobs below. If `IMAGES_DIR` is empty or missing, falls back to random tensors so the timing still works.

In [19]:
IMAGES_DIR = Path('./images')   # folder with .jpg / .png / .jpeg
IMG_SIZE   = 1280
BATCH_SIZE = 1

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
model_dev = model.to(DEVICE)


def load_batch(paths, size):
    arrs = [
        np.asarray(Image.open(p).convert('RGB').resize((size, size)),
                   dtype=np.float32) / 255.0
        for p in paths
    ]
    return torch.from_numpy(np.stack(arrs)).permute(0, 3, 1, 2)  # NCHW


img_paths = sorted(
    p for p in (IMAGES_DIR.glob('*') if IMAGES_DIR.is_dir() else [])
    if p.suffix.lower() in ('.jpg', '.jpeg', '.png')
)
if img_paths:
    print(f'found {len(img_paths)} images in {IMAGES_DIR}')
    n_batches = len(img_paths) // BATCH_SIZE
else:
    print(f'no images in {IMAGES_DIR} — using random tensors')
    n_batches = 8

# warmup (excluded from timing — JIT, cudnn autotune, malloc caches, …)
with torch.inference_mode():
    warm = torch.zeros(BATCH_SIZE, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    for _ in range(2):
        _ = model_dev(warm)

times_ms = []
with torch.inference_mode():
    for i in range(n_batches):
        if img_paths:
            x = load_batch(img_paths[i*BATCH_SIZE:(i+1)*BATCH_SIZE], IMG_SIZE).to(DEVICE)
        else:
            x = torch.randn(BATCH_SIZE, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
        if DEVICE == 'cuda': torch.cuda.synchronize()
        t0 = time.perf_counter()
        _ = model_dev(x)
        if DEVICE == 'cuda': torch.cuda.synchronize()
        times_ms.append((time.perf_counter() - t0) * 1000)

med = statistics.median(times_ms)
print(f'\ndevice:     {DEVICE}')
print(f'img_size:   {IMG_SIZE}px')
print(f'batch:      {BATCH_SIZE}')
print(f'batches:    {len(times_ms)}')
print(f'median:     {med:.1f} ms / batch')
print(f'min / max:  {min(times_ms):.1f} / {max(times_ms):.1f} ms')
print(f'throughput: {1000 * BATCH_SIZE / med:.1f} img/sec')

found 56 images in images

device:     cuda
img_size:   1280px
batch:      1
batches:    56
median:     107.4 ms / batch
min / max:  107.1 / 111.4 ms
throughput: 9.3 img/sec


## 3. Export to ONNX

`torch.onnx.export` traces the model with a dummy input and writes a self-contained `.onnx` file — graph + weights, no Python class reference. Dynamic axes are declared on the batch and spatial dims so the same file works for any `(batch, H, W)` you pick in Streamlit later.

In [ ]:
MODEL_NAME = 'mini_unet_v1'
ONNX_PATH = Path.cwd() / f'{MODEL_NAME}.onnx'

dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE)
torch.onnx.export(
    model.cpu().eval(),
    dummy,
    str(ONNX_PATH),
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={
        'input':  {0: 'batch', 2: 'h', 3: 'w'},
        'output': {0: 'batch', 2: 'h', 3: 'w'},
    },
    opset_version=17
    # legacy tracer — avoids the onnxscript dep that the new dynamo exporter needs
)
print(f'saved → {ONNX_PATH}  ({ONNX_PATH.stat().st_size/1024:.0f} KB)')